In [3]:
!pip install openai
!pip install gradio
!pip install chromadb

In [1]:
import os
os.environ['API_GATEWAY_KEY'] = "Pa1a8NxLVl5rG2DgbtDj"

In [2]:
import os
from openai import OpenAI

OPENAI_API_KEY = os.environ["API_GATEWAY_KEY"]

client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any value",
    default_headers={"x-api-key": OPENAI_API_KEY},
)

In [4]:
import os
os.makedirs("chroma_db", exist_ok=True)

In [5]:
RESTRICTED_TOPICS = [
    "cat", "dog",
    "horoscope", "zodiac",
    "taylor swift"
]

def guardrails(user_message):
    lower = user_message.lower()

    # Topic restriction
    for topic in RESTRICTED_TOPICS:
        if topic in lower:
            return "I'm not allowed to discuss that topic."

    # System prompt protection
    if "system prompt" in lower or "ignore previous instructions" in lower:
        return "I cannot reveal or modify my internal instructions."

    return None

In [6]:
import requests

def weather_service():
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": 43.7,
        "longitude": -79.4,
        "current_weather": True
    }

    r = requests.get(url, params=params)
    data = r.json()

    temp = data["current_weather"]["temperature"]
    wind = data["current_weather"]["windspeed"]

    # Transform output (NOT verbatim)
    return f"In Toronto right now, it's about {temp}°C with winds moving at {wind} km/h."

In [9]:
import chromadb
from chromadb.utils import embedding_functions
import os

chroma_client = chromadb.PersistentClient(path="./chroma_db")

BASE_URL = "https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1" # Define BASE_URL here

embedding_function = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.environ["API_GATEWAY_KEY"],
    model_name="text-embedding-3-small",
    api_base=BASE_URL  # ☀ Important for gateway
)

collection = chroma_client.get_or_create_collection(
    name="knowledge_base",
    embedding_function=embedding_function
)

# Add small dataset (run once)
collection.add(
    documents=[
        "Artificial Intelligence is the simulation of human intelligence.",
        "Machine learning is a subset of AI.",
        "Neural networks are inspired by biological neurons.",
        "Deep learning uses multi-layer neural networks."
    ],
    ids=["1","2","3","4"]
)

PermissionDeniedError: Error code: 403 - {'message': 'Forbidden'} in add.

In [10]:
def semantic_search(query):
    results = collection.query(
        query_texts=[query],
        n_results=2
    )

    context = "\n".join(results["documents"][0])

    prompt = f"""
    Use this context to answer the question:

    {context}

    Question: {query}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [11]:
def calculate(expression):
    try:
        return str(eval(expression))
    except:
        return "Invalid expression."

In [12]:
import gradio as gr

SYSTEM_PROMPT = """
You are Nova, a witty but professional AI assistant.
You are concise, friendly, and insightful.
"""

conversation_history = []

def main_router(user_input):

    # 1️⃣ Guardrails first
    blocked = guardrails(user_input)
    if blocked:
        return blocked

    lower = user_input.lower()

    # 2️⃣ Service 1
    if "weather" in lower:
        return weather_service()

    # 3️⃣ Service 3
    elif user_input.startswith("calculate"):
        expr = user_input.replace("calculate", "").strip()
        return calculate(expr)

    # 4️⃣ Service 2
    elif "ai" in lower or "machine learning" in lower:
        return semantic_search(user_input)

    # 5️⃣ Default LLM with memory
    else:
        conversation_history.append({"role": "user", "content": user_input})

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "system", "content": SYSTEM_PROMPT}] + conversation_history
        )

        reply = response.choices[0].message.content

        conversation_history.append({"role": "assistant", "content": reply})

        # Optional memory limit
        if len(conversation_history) > 10:
            conversation_history.pop(0)

        return reply

In [13]:
def respond(message, chat_history):
    reply = main_router(message)
    chat_history.append((message, reply))
    return "", chat_history

with gr.Blocks() as demo:
    gr.Markdown("# 🤖 Nova AI Assistant")
    chatbot = gr.Chatbot()
    msg = gr.Textbox()
    clear = gr.Button("Clear")

    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: None, None, chatbot)

demo.launch()

/tmp/ipython-input-4098671123.py:8: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()
/tmp/ipython-input-4098671123.py:8: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4e586b0d18b2252372.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
